# Advanced RAG Patterns — Runnable
Your Specialist Track covers these as concept notes. This notebook makes **CRAG, Adaptive, Multi-hop, and Fusion RAG** actually executable so you can watch them behave. Runs fully offline with a mock embedder + mock LLM — swap in your real `InHouseEmbeddings` and `ask()` to make it real.

In [ ]:
# Self-contained mock embedder + LLM so this notebook runs with NO api key / NO network.
# Swap MockEmbedder -> your InHouseEmbeddings and mock_llm -> your ask() for real use.
import numpy as np, re
from collections import Counter

class MockEmbedder:
    """Deterministic bag-of-words embedding: same words -> similar vectors.
    Good enough to demonstrate retrieval behavior without a real model."""
    def __init__(self, dim=64):
        self.dim = dim
    def _vec(self, text):
        rng = np.random.default_rng(0)
        base = {}
        v = np.zeros(self.dim)
        for w in re.findall(r"\w+", text.lower()):
            h = abs(hash(w)) % self.dim
            v[h] += 1.0
        n = np.linalg.norm(v)
        return v / n if n > 0 else v
    def embed_documents(self, texts): return [self._vec(t).tolist() for t in texts]
    def embed_query(self, text): return self._vec(text).tolist()

embedder = MockEmbedder()

def mock_llm(system, user, **kw):
    """Extremely dumb stand-in: echoes retrieved context if present, else says IDK.
    Replace with your real ask(system, user, model=...) helper for genuine answers."""
    if "no relevant" in user.lower() or "context: \n\nquestion" in user.lower():
        return "I don't know based on the provided context."
    # pull the 'Context:' block back out as a fake 'answer'
    m = re.search(r"Context:(.*?)Question:", user, re.S)
    ctx = m.group(1).strip() if m else ""
    return f"(mock answer grounded in retrieved context) {ctx[:160]}"

## Shared tiny knowledge base + retriever

In [ ]:
import numpy as np

DOCS = {
    "auth_a": "API A uses OAuth2 bearer tokens. Rate limit is 100 requests per minute.",
    "auth_b": "API B uses static API keys. Rate limit is 50 requests per minute.",
    "refund": "Refunds are processed within 5 business days to the original payment method.",
    "ship":   "Standard shipping takes 3 to 5 business days. Express takes 1 day.",
}
ids = list(DOCS); texts = list(DOCS.values())
mat = np.array(embedder.embed_documents(texts))

def retrieve(query, k=2):
    q = np.array(embedder.embed_query(query))
    sims = mat @ q
    order = np.argsort(sims)[::-1][:k]
    return [(ids[i], texts[i], float(sims[i])) for i in order]

for r in retrieve("what auth does API A use"): print(r)

## 1. Corrective RAG (CRAG) — check retrieval quality, fall back if weak

In [ ]:
def crag(query, k=2, min_sim=0.15):
    hits = retrieve(query, k)
    best_sim = hits[0][2] if hits else 0.0
    if best_sim < min_sim:
        # corrective action: retrieval too weak -> don't answer from it
        return f"[CRAG] top similarity {best_sim:.2f} < {min_sim}; refusing to answer, would rewrite query or widen search."
    context = "\n".join(h[1] for h in hits)
    return mock_llm("Answer only from context.", f"Context:\n{context}\nQuestion: {query}")

print(crag("what auth does API A use"))          # good retrieval
print(crag("what is the airspeed of a swallow"))  # weak retrieval -> corrective action

## 2. Adaptive RAG — route by query type before retrieving

In [ ]:
def classify(query):
    q = query.lower()
    if any(w in q for w in ["compare", "vs", "difference", "both"]): return "comparison"
    if any(w in q for w in ["and", "then", "after"]):                return "multi_part"
    return "simple"

def adaptive_rag(query):
    kind = classify(query)
    if kind == "comparison":
        # comparison queries benefit from more chunks
        hits = retrieve(query, k=4)
    elif kind == "multi_part":
        hits = retrieve(query, k=3)
    else:
        hits = retrieve(query, k=1)
    context = "\n".join(h[1] for h in hits)
    return kind, mock_llm("Answer only from context.", f"Context:\n{context}\nQuestion: {query}")

for q in ["what auth does API A use", "compare API A and API B rate limits"]:
    kind, ans = adaptive_rag(q)
    print(f"[{kind}] {q}\n  -> {ans}\n")

## 3. Multi-hop RAG — decompose, retrieve per sub-question, combine

In [ ]:
def multi_hop(query, sub_questions):
    all_context = []
    for sq in sub_questions:
        hits = retrieve(sq, k=1)
        all_context.append(f"[{sq}] {hits[0][1]}")
    context = "\n".join(all_context)
    return mock_llm("Answer only from context.", f"Context:\n{context}\nQuestion: {query}")

print(multi_hop(
    "Compare the auth and rate limits of API A and API B",
    ["auth method of API A", "rate limit of API A", "auth method of API B", "rate limit of API B"],
))

## 4. Fusion RAG — multiple query variations, fuse with Reciprocal Rank Fusion

In [ ]:
def rrf_fuse(ranked_lists, k=60):
    scores = {}
    for lst in ranked_lists:
        for rank, (doc_id, _, _) in enumerate(lst):
            scores[doc_id] = scores.get(doc_id, 0) + 1.0 / (rank + k)
    return sorted(scores.items(), key=lambda x: -x[1])

variations = ["API A authentication", "how does API A log in", "API A oauth token"]
ranked_lists = [retrieve(v, k=3) for v in variations]
fused = rrf_fuse(ranked_lists)
print("Fused ranking (doc_id, rrf_score):")
for doc_id, score in fused:
    print(f"  {doc_id}: {score:.4f}")

## Your turn
Replace `MockEmbedder` with your real `InHouseEmbeddings` and `mock_llm` with your `ask()` helper, point `DOCS` at real content, and re-run. The pattern logic doesn't change at all — only the embedding + generation calls do. That separation (patterns vs. model plumbing) is itself an expert-level insight.